# Lesson 26 Lab — Accuracy Recovery, Rollback, and Slice Error Analysis

**Puzzle:** What should happen when overall accuracy recovers but one long-tail class remains below its rollback threshold?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Recovery is not one scalar. A pruned model may recover aggregate accuracy by favoring common classes while a rare or high-risk class remains degraded. The release artifact must preserve the dense baseline, pruned checkpoint identity, confusion matrix, slice deltas, acceptance thresholds, and deterministic rollback decision.


## 0. Predict before running

1. Predict which class contributes least to aggregate accuracy.
2. Compute recall from one confusion-matrix row.
3. Write a two-part acceptance rule that protects the tail.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

An imbalanced three-class CUDA classification task, dense checkpoint, magnitude-pruned candidate, short recovery, confusion matrices, per-class recall, aggregate accuracy, and a frozen worst-class gate form the experiment.

- Aggregate recovery can conceal minority-class regression.
- Thresholds must be frozen before the candidate is evaluated.
- Rollback is a derived decision linked to an immutable baseline.


## 2. Derive the mechanism

Accuracy weights each example equally, so a 5% class can fall sharply while changing the aggregate by less than one point. Per-class recall `TP_c/(TP_c+FN_c)` and a confusion matrix expose the shift. The gate can require both `accuracy_drop <= a` and `min recall_drop >= -r`. Recovery selects the best checkpoint only on validation criteria; the rollback revision remains immutable.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 26
LESSON_TITLE = 'Accuracy Recovery, Rollback, and Slice Error Analysis'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260834
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | dense checkpoint with frozen validation confusion matrix |
| Candidate | 70%-pruned recovered checkpoint evaluated under the same split |
| Held constant | dataset, imbalance, checkpoint, mask, recovery steps, seed, thresholds, and evaluation code |
| Measurements | aggregate accuracy, per-class recall, worst recall drop, confusion matrices, and rollback decision |
| Evidence | `pytorch-gpu` |

**Experiment:** Prune and recover an imbalanced classifier, then derive release or rollback from aggregate and per-class gates.


## 5. Read the experiment code

The notebook trains a compact baseline, freezes it, prunes a clone, and records both immediate and recovered metrics. Confusion matrices are computed explicitly on GPU predictions and stored as lists. The final decision is programmatically derived from the predeclared aggregate and tail thresholds.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
d,c=20,3; counts=[900,250,90]; centers=torch.randn(c,d,device=DEVICE)*2.0
xs=[]; ys=[]
for cls,count in enumerate(counts): xs.append(centers[cls]+torch.randn(count,d,device=DEVICE)); ys.append(torch.full((count,),cls,device=DEVICE,dtype=torch.long))
train_x=torch.cat([z[:int(len(z)*0.8)] for z in xs]); train_y=torch.cat([z[:int(len(z)*0.8)] for z in ys]); val_x=torch.cat([z[int(len(z)*0.8):] for z in xs]); val_y=torch.cat([z[int(len(z)*0.8):] for z in ys])
model=nn.Sequential(nn.Linear(d,48),nn.ReLU(),nn.Linear(48,c)).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=0.025)
for step in range(180):
    idx=torch.randint(0,train_x.shape[0],(96,),device=DEVICE); opt.zero_grad(); loss=F.cross_entropy(model(train_x[idx]),train_y[idx]); loss.backward(); opt.step()
def evaluate(m):
    m.eval()
    with torch.inference_mode(): pred=m(val_x).argmax(1)
    cm=torch.zeros(c,c,device=DEVICE,dtype=torch.int64)
    for t,p in zip(val_y,pred): cm[t,p]+=1
    recall=(cm.diag()/cm.sum(1).clamp_min(1)).float(); return float((pred==val_y).float().mean().item()),recall,cm
dense_acc,dense_rec,dense_cm=evaluate(model); pruned=copy.deepcopy(model); masks={}
with torch.no_grad():
    for name,p in pruned.named_parameters():
        if "weight" in name: masks[p]=magnitude_mask(p,0.70); p.mul_(masks[p])
im_acc,im_rec,im_cm=evaluate(pruned); opt=torch.optim.Adam(pruned.parameters(),lr=0.008)
for step in range(70):
    idx=torch.randint(0,train_x.shape[0],(96,),device=DEVICE); opt.zero_grad(); loss=F.cross_entropy(pruned(train_x[idx]),train_y[idx]); loss.backward(); opt.step()
    with torch.no_grad():
        for p,m in masks.items(): p.mul_(m)
rec_acc,rec_rec,rec_cm=evaluate(pruned); drops=rec_rec-dense_rec; rollback=bool((dense_acc-rec_acc)>0.03 or drops.min().item()<-0.10)
metrics={"dense_accuracy":dense_acc,"pruned_immediate_accuracy":im_acc,"recovered_accuracy":rec_acc,"dense_recall":dense_rec.cpu().tolist(),"immediate_recall":im_rec.cpu().tolist(),"recovered_recall":rec_rec.cpu().tolist(),"worst_recall_drop":float(drops.min().item()),"dense_confusion":dense_cm.cpu().tolist(),"recovered_confusion":rec_cm.cpu().tolist(),"accuracy_drop_threshold":0.03,"recall_drop_threshold":-0.10,"rollback_required":rollback,"final_sparsity":sum((p==0).sum().item() for p in masks)/sum(p.numel() for p in masks)}
analysis=(f"Dense, immediate-pruned, and recovered aggregate accuracies were {dense_acc:.1%}, {im_acc:.1%}, and {rec_acc:.1%}. "
          f"The worst class-recall change after recovery was {metrics['worst_recall_drop']:.1%}; under the frozen 3-point aggregate "
          f"and 10-point recall gates, rollback_required={rollback}.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Dense accuracy | 100.00% |
| Pruned immediate accuracy | 99.19% |
| Recovered accuracy | 100.00% |
| Worst recall drop | 0.00% |
| Rollback required | no |
| Final sparsity | 70.02% |


## 7. Interpret rather than merely print

Dense, immediate-pruned, and recovered aggregate accuracies were 100.0%, 99.2%, and 100.0%. The worst class-recall change after recovery was 0.0%; under the frozen 3-point aggregate and 10-point recall gates, rollback_required=False.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 26,
    "title": 'Accuracy Recovery, Rollback, and Slice Error Analysis',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Recovery is complete only when aggregate and protected slices pass a predeclared gate with a tested rollback path.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 26,
  "title": "Accuracy Recovery, Rollback, and Slice Error Analysis",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260834
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "dense_accuracy": 1.0,
    "pruned_immediate_accuracy": 0.9919354319572449,
    "recovered_accuracy": 1.0,
    "dense_recall": [
      1.0,
      1.0,
      1.0
    ],
    "immediate_recall": [
      0.9888888597488403,
      1.0,
      1.0
    ],
    "recovered_recall": [
      1.0,
      1.0,
      1.0
    ],
    "worst_recall_drop": 0.0,
    "dense_confusion": [
      [
        180,
        0,
        0
      ],
      [
        0,
        50,
        0
      ],
      [
        0,
        0,
        18
      ]
    ],
    "recovered_confusion": [
      [
        180,
        0,
        0
      ],
      [
        0,
        50,
        0
      ],
      [
    

## 9. Make the bounded decision

> Recovery is complete only when aggregate and protected slices pass a predeclared gate with a tested rollback path.

**Acceptance/rollback:** Release only when every frozen aggregate and critical-slice gate passes; otherwise select the recorded dense revision as rollback.

**Failure analysis:** A toy class imbalance is not a production taxonomy, and recall alone ignores precision or calibration. Choosing thresholds after seeing the candidate invalidates the gate. Recovery on the final test split leaks evaluation.


## 10. Extend the evidence

Add precision, calibration, and domain slices; separate train/validation/test; rehearse the actual model-registry rollback and monitor the same gates during canary.

The full evidence boundary and references are in [`README.md`](README.md).
